In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

# ==========================================
# 1. KASUTAJALIIDES
# ==========================================

style = {'description_width': '140px'}
l_layout = widgets.Layout(width='250px')

b_in = widgets.IntText(value=275, description='Laius b (mm):', style=style, layout=l_layout)
h_in = widgets.IntText(value=450, description='Kõrgus h (mm):', style=style, layout=l_layout)
Es_in = widgets.FloatText(value=200000, description='E_s (N/mm²):', style=style, layout=l_layout)
Ecm_in = widgets.FloatText(value=31000, description='E_cm (N/mm²):', style=style, layout=l_layout)
fctm_in = widgets.FloatText(value=2.60, description='f_ctm (N/mm²):', style=style, layout=l_layout)

As1_in = widgets.IntText(value=402, description='A_s1 tõmme (mm²):', style=style, layout=l_layout)
As2_in = widgets.IntText(value=0, description='A_s2 surve (mm²):', style=style, layout=l_layout)
ds1_in = widgets.IntText(value=415, description='d_s1 (mm):', style=style, layout=l_layout)
ds2_in = widgets.IntText(value=0, description='d_s2 (mm):', style=style, layout=l_layout)

N_in = widgets.FloatText(value=0.00, description='Pikijõud N (kN):', style=style, layout=l_layout)
M_in = widgets.FloatText(value=51.30, description='Moment M (kNm):', style=style, layout=l_layout)
fi_in = widgets.FloatText(value=2.2, description='Roometegur φ:', style=style, layout=l_layout)

vasak_tulp = widgets.VBox([
    widgets.HTML("<h3 style='margin-top:0;color:#2c3e50;'>Eurokoodi sisendid</h3>"),
    widgets.HTML("<h4>Geomeetria</h4>"),
    b_in, h_in, Es_in, Ecm_in, fctm_in,
    widgets.HTML("<h4>Sarrustus</h4>"),
    As1_in, As2_in, ds1_in, ds2_in,
    widgets.HTML("<h4>Koormused</h4>"),
    N_in, M_in, fi_in
], layout=widgets.Layout(
    padding='15px',
    border='1px solid #ccc',
    background_color='#f9f9f9',
    min_width='280px'
))

# Loome väljundi vidina, et vältida HTML ja Markdown elementide segi paiskamist
valjund = widgets.Output()

# ==========================================
# 2. EUROKOODI ARVUTUSED JA RAPORT
# ==========================================

def täielik_eurokoodi_arvutus(
    b, h, Es, Ecm, fctm,
    As1, As2, ds1, ds2,
    N_v, M_v, fi
):
    # Puhastame eelmise arvutuse tulemused väljundist
    valjund.clear_output(wait=True)
    
    with valjund:
        N = N_v * 10**3
        M = M_v * 10**6

        Ac = b * h
        Soc = b * h * (h / 2)
        Ic = (b * h**3) / 12

        # --- STAADIUM I: pragunemata ristlõige enne roomet ---
        alpha_s1 = Es / Ecm
        Ared1 = Ac + alpha_s1 * As1 + alpha_s1 * As2

        S0red1 = (
            Soc
            + alpha_s1 * As1 * (h - ds1)
            + alpha_s1 * As2 * (h - ds2)
        )

        y0red1 = S0red1 / Ared1 if Ared1 != 0 else h / 2
        x1 = h - y0red1

        ys1_1 = ds1 - x1
        ys2_1 = y0red1 - ds2

        Ired1 = (
            Ic
            + Ac * (y0red1 - h / 2)**2
            + alpha_s1 * As1 * ys1_1**2
            + alpha_s1 * As2 * ys2_1**2
        )

        Wcred1 = Ired1 / x1 if x1 != 0 else 0
        Wtred1 = Ired1 / y0red1 if y0red1 != 0 else 0

        sigma_c_max1 = -N / Ared1 - M / Wcred1 if Wcred1 else 0
        sigma_ct_max1 = -N / Ared1 + M / Wtred1 if Wtred1 else 0
        Mcr1 = fctm * Wtred1 / 10**6

        # --- STAADIUM II: pragunemata ristlõige pärast roomet ---
        Ec_ef = (1.05 * Ecm) / (1 + fi)
        alpha_s2 = Es / Ec_ef

        Ared2 = Ac + alpha_s2 * As1 + alpha_s2 * As2

        S0red2 = (
            Soc
            + alpha_s2 * As1 * (h - ds1)
            + alpha_s2 * As2 * (h - ds2)
        )

        y0red2 = S0red2 / Ared2 if Ared2 != 0 else h / 2
        x2 = h - y0red2

        Ired2 = (
            Ic
            + Ac * (y0red2 - h / 2)**2
            + alpha_s2 * As1 * (ds1 - x2)**2
            + alpha_s2 * As2 * (x2 - ds2)**2
        )

        Wcred2 = Ired2 / x2 if x2 != 0 else 0
        Wtred2 = Ired2 / y0red2 if y0red2 != 0 else 0

        sigma_c_max2 = -N / Ared2 - M / Wcred2 if Wcred2 else 0
        sigma_ct_max2 = -N / Ared2 + M / Wtred2 if Wtred2 else 0
        Mcr2 = fctm * Wtred2 / 10**6

        # --- STAADIUM III: pragunenud ristlõige enne roomet ---
        A_eff3 = alpha_s1 * (As1 + As2)
        B_eff3 = 2 * alpha_s1 * (As1 * ds1 + As2 * ds2)

        x_cr3 = (
            (-A_eff3 + (A_eff3**2 + b * B_eff3)**0.5) / b
            if b != 0 else 0
        )

        Icr3 = (
            (b * x_cr3**3) / 3
            + alpha_s1 * As1 * (ds1 - x_cr3)**2
            + alpha_s1 * As2 * (x_cr3 - ds2)**2
        )

        sigma_c_max3 = -M * x_cr3 / Icr3 if Icr3 else 0
        sigma_s1_3 = (
            alpha_s1 * M * (ds1 - x_cr3) / Icr3
            if Icr3 else 0
        )

        # --- STAADIUM IV: pragunenud ristlõige pärast roomet ---
        A_eff4 = alpha_s2 * (As1 + As2)
        B_eff4 = 2 * alpha_s2 * (As1 * ds1 + As2 * ds2)

        x_cr4 = (
            (-A_eff4 + (A_eff4**2 + b * B_eff4)**0.5) / b
            if b != 0 else 0
        )

        Icr4 = (
            (b * x_cr4**3) / 3
            + alpha_s2 * As1 * (ds1 - x_cr4)**2
            + alpha_s2 * As2 * (x_cr4 - ds2)**2
        )

        sigma_c_max4 = -M * x_cr4 / Icr4 if Icr4 else 0
        sigma_s1_4 = (
            alpha_s2 * M * (ds1 - x_cr4) / Icr4
            if Icr4 else 0
        )

        # ==========================================
        # 3. RAPORTI KUVAMINE
        # ==========================================

        display(HTML('<div style="font-family:Arial,sans-serif;padding-left:10px;"><h3 style="color:#2c3e50;">1. PRAGUNEMATA RISTLÕIGE ENNE ROOMET</h3></div>'))
        display(Markdown(rf"""
$$ \alpha_s = \frac{{E_s}}{{E_{{cm}}}} = \frac{{{Es:.0f}}}{{{Ecm:.0f}}} = {alpha_s1:.3f} $$
$$ A_{{red}} = {b}\cdot {h} + {alpha_s1:.3f}\cdot ({As1}+{As2}) = {Ared1:.0f}\ \mathrm{{mm}}^2 $$
$$ S_{{0,red}} = {S0red1/10**6:.3f}\cdot 10^6\ \mathrm{{mm}}^3 $$
$$ y_{{0,red}} = \frac{{{S0red1/10**6:.3f}\cdot 10^6}}{{{Ared1:.0f}}} = {y0red1:.2f}\ \mathrm{{mm}} $$
$$ x = h-y_{{0,red}} = {h}-{y0red1:.2f} = {x1:.2f}\ \mathrm{{mm}} $$
$$ I_{{red}} = {Ired1/10**6:.3f}\cdot 10^6\ \mathrm{{mm}}^4 $$
$$ M_{{cr}} = {Mcr1:.3f}\ \mathrm{{kNm}} $$
"""))

        display(HTML('<h3 style="color:#2c3e50;padding-left:10px;">2. PRAGUNEMATA RISTLÕIGE PÄRAST ROOMET</h3>'))
        display(Markdown(rf"""
$$ E_{{c,ef}} = \frac{{1.05\cdot {Ecm:.0f}}}{{1+{fi}}} = {Ec_ef:.2f}\ \mathrm{{N/mm}}^2 $$
$$ \alpha_s = \frac{{E_s}}{{E_{{c,ef}}}} = {alpha_s2:.3f} $$
$$ y_{{0,red,t}} = {y0red2:.2f}\ \mathrm{{mm}},\qquad x_t = {x2:.2f}\ \mathrm{{mm}} $$
$$ I_{{red,t}} = {Ired2/10**6:.3f}\cdot 10^6\ \mathrm{{mm}}^4 $$
$$ M_{{cr,t}} = {Mcr2:.3f}\ \mathrm{{kNm}} $$
"""))

        display(HTML('<h3 style="color:#c62828;padding-left:10px;">3. PRAGUNENUD RISTLÕIGE ENNE ROOMET</h3>'))
        display(Markdown(rf"""
$$ x_{{cr}} = \frac{{-A_{{eff}}+\sqrt{{A_{{eff}}^2+bB_{{eff}}}}}}{{b}} = {x_cr3:.2f}\ \mathrm{{mm}} $$
$$ I_{{cr,red}} = {Icr3/10**6:.3f}\cdot 10^6\ \mathrm{{mm}}^4 $$
"""))

        display(HTML('<h3 style="color:#c62828;padding-left:10px;">4. PRAGUNENUD RISTLÕIGE PÄRAST ROOMET</h3>'))
        display(Markdown(rf"""
$$ x_{{cr,t}} = {x_cr4:.2f}\ \mathrm{{mm}} $$
$$ I_{{cr,red,t}} = {Icr4/10**6:.3f}\cdot 10^6\ \mathrm{{mm}}^4 $$
"""))

        display(HTML(f"""
<h3 style="color:#1e6823;padding-left:10px;">KOKKUVÕTE: PINGETE KOONDTABEL [MPa]</h3>
<div style="overflow-x:auto;padding-left:10px;">
<table border="1" cellpadding="6" style="border-collapse:collapse;text-align:center;width:100%;font-family:Arial,sans-serif;">
<tr style="background:#2e7d32;color:white;">
<th>Pinge / parameeter</th>
<th>Pragunemata enne</th>
<th>Pragunemata pärast</th>
<th>Pragunenud enne</th>
<th>Pragunenud pärast</th>
</tr>
<tr>
<td><b>σ_c,max</b> – betooni surve</td>
<td>{sigma_c_max1:.2f}</td>
<td>{sigma_c_max2:.2f}</td>
<td>{sigma_c_max3:.2f}</td>
<td>{sigma_c_max4:.2f}</td>
</tr>
<tr>
<td><b>σ_ct,max</b> – betooni tõmme</td>
<td>{sigma_ct_max1:.2f}</td>
<td>{sigma_ct_max2:.2f}</td>
<td>–</td>
<td>–</td>
</tr>
<tr>
<td><b>σ_s1</b> – tõmbearmatuur</td>
<td>–</td>
<td>–</td>
<td>{sigma_s1_3:.2f}</td>
<td>{sigma_s1_4:.2f}</td>
</tr>
</table>
</div>
"""))

# ==========================================
# 4. KÄIVITAMINE JA SIDUMINE
# ==========================================

keskosa = widgets.interactive_output(
    täielik_eurokoodi_arvutus,
    {
        'b': b_in,
        'h': h_in,
        'Es': Es_in,
        'Ecm': Ecm_in,
        'fctm': fctm_in,
        'As1': As1_in,
        'As2': As2_in,
        'ds1': ds1_in,
        'ds2': ds2_in,
        'N_v': N_in,
        'M_v': M_in,
        'fi': fi_in
    }
)

# Kuvame sisendid ja fikseeritud väljundi kõrvuti või üksteise all
layout_paigutus = widgets.HBox([vasak_tulp, valjund])
display(layout_paigutus)
